# Hard Case: Complex Pagination

Many sites split their data across **many pages**. Two common patterns:

1. **A "Next" button that is a plain link** → just use `requests` + BeautifulSoup. Instead of
   guessing the number of pages, we **follow the Next button until it disappears**.
2. **A "Next" button driven by JavaScript** (needs a click, the URL may not change) → use **Selenium**.

We'll practice on `https://quotes.toscrape.com` (there's a Next button on every page).

> Ground rule: if you can do it with `requests` (approach 1), **don't** rush to Selenium.
> Use Selenium only when JavaScript/interaction is required.

**Tooling:** `requests` + `beautifulsoup4` (approach 1), `selenium` + `WebDriverWait` (approach 2).


## Approach 1 — Follow the "Next" button with `requests` (recommended)

The pattern: scrape a page → look for the Next element (`<li class="next"><a href="...">`) →
if it exists, move on to that URL; if not, **stop**. Use `urljoin` so that a
relative URL (`/page/2/`) becomes a full URL. Always set a **maximum page limit**
as a safeguard against an endless loop.


In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://quotes.toscrape.com/"


def scrape_all_pages(start_url=BASE, max_pages=20):
    url = start_url
    all_quotes = []
    page = 0
    while url and page < max_pages:  # max_pages = safeguard
        page += 1
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        for q in soup.find_all("div", class_="quote"):
            all_quotes.append(q.find("span", class_="text").get_text(strip=True))

        # look for the Next button; if missing -> url becomes None -> loop stops
        next_li = soup.find("li", class_="next")
        url = urljoin(BASE, next_li.find("a")["href"]) if next_li else None
        print(f"page {page:>2} -> total collected: {len(all_quotes)}")

    return all_quotes


quotes = scrape_all_pages()
print("\nDone. Total quotes:", len(quotes))


page  1 -> total collected: 10


page  2 -> total collected: 20


page  3 -> total collected: 30


page  4 -> total collected: 40


page  5 -> total collected: 50


page  6 -> total collected: 60


page  7 -> total collected: 70


page  8 -> total collected: 80


page  9 -> total collected: 90


page 10 -> total collected: 100

Done. Total quotes: 100


## Approach 2 — Click the "Next" button with Selenium

If the Next button needs JavaScript / a click (the URL doesn't always change), we use Selenium.
The pattern is the same: collect data → find the Next button → click it → repeat until the Next button disappears.

Use `WebDriverWait` to **wait** for the quotes to appear first (don't blindly `time.sleep`),
and `try/except NoSuchElementException` to detect the last page.


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException


def make_driver(headless=True):
    o = Options()
    if headless:  # set False if you want to watch the browser
        o.add_argument("--headless=new")
    o.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=o)


driver = make_driver()
all_quotes = []
try:
    driver.get("https://quotes.toscrape.com/js/")
    page = 0
    while page < 20:  # safeguard
        page += 1
        # wait for the (JS-rendered) quotes to appear
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".quote .text"))
        )
        for el in driver.find_elements(By.CSS_SELECTOR, ".quote .text"):
            all_quotes.append(el.text)

        # look for the Next button; missing -> last page
        try:
            driver.find_element(By.CSS_SELECTOR, "li.next a").click()
        except NoSuchElementException:
            break

    print(f"Finished on page {page}, total {len(all_quotes)} quotes")
finally:
    driver.quit()


Finished on page 10, total 100 quotes


## Conclusion & Exercise

- **Following the Next button until it disappears** is more robust than hardcoding `range(1, N)`
  (if the number of pages changes, the code still works).
- Always set a **maximum page limit** as a safeguard.
- Reach for `requests` first; use Selenium only when Next requires JavaScript/a click.

**Exercise:** modify approach 1 so that each quote stores **text + author** (a dict), then count
how many quotes there are per author. A sample solution is in the next cell.


In [3]:
# Sample solution to the exercise
from collections import Counter


def scrape_with_author(start_url=BASE, max_pages=20):
    url, out, page = start_url, [], 0
    while url and page < max_pages:
        page += 1
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for q in soup.find_all("div", class_="quote"):
            out.append(
                {
                    "text": q.find("span", class_="text").get_text(strip=True),
                    "author": q.find("small", class_="author").get_text(strip=True),
                }
            )
        nx = soup.find("li", class_="next")
        url = urljoin(BASE, nx.find("a")["href"]) if nx else None
    return out


data = scrape_with_author()
print("Total items:", len(data))
print("Top 5 authors:", Counter(d["author"] for d in data).most_common(5))


Total items: 100
Top 5 authors: [('Albert Einstein', 10), ('J.K. Rowling', 9), ('Marilyn Monroe', 7), ('Dr. Seuss', 6), ('Mark Twain', 6)]
